# Long-form visual evidence audit

## What you will build

Compare a bounded maintenance claim with two synthetic inspection drawings and a complete,
extended equipment history. Four independent `Noul` propositions distinguish visual support,
visual contradiction, report support, and report contradiction. Python creates a dated audit
entry or sends it to [review](../../docs/glossary.md#review), meaning a person must settle the
same claim. All entries and queues are simulated.

The drawings are illustrative fixtures, not photographs of real equipment. Each of the 26 cases
has its own 44-entry report, a historical drawing, and a current drawing. Archived faults appear
near the beginning; current sign-offs and contradictory addenda appear near the end. Internal
bearing service cannot be inferred from a closed exterior. Even an accepted `image_supported`
claim certifies only the stated visible condition at the supplied cutoff, never equipment safety.

Sources:

- S02: [TypeSafe AI: Primitives](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S09: [TypeSafe AI: Images](https://docs.typesafe.ai/images)
- S11: [TypeSafe AI: Models and long context](https://docs.typesafe.ai/models)
- S12: [TypeSafe AI: Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)

## Setup and run mode

Offline replay is keyless and makes no network calls. `images=True` and `long_context=True`
are independent live opt-ins: the former enables `TypeSafe-Danger-Zone-Images: true`, the latter
`TypeSafe-Danger-Zone-Long-Context: true`. The pinned requested model is `jev-1.14.0`.
Enabling long context does not measure model capacity. This is an extended-report demonstration,
not a 262K-token benchmark; no tokenizer-derived or invented token totals appear.

The local request budget is 26, one request per case with all four independent questions batched.
The live transport also has its own attempt budget; retries count there. The notebook stores and
reuses each result rather than making another request when showing it later.

In [ ]:
from IPython.display import display
from PIL import Image

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import confusion_matrix, evaluate_outcomes
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.simulation import ActionLog, ReviewQueue
from jev_cookbook.style import apply_style, plot_confusion_matrix, run_header, show_answer
from jev_cookbook.vision import run_record

apply_style()
helpers = load_helpers()
examples, labels = load_inputs(), load_labels()
backend = get_backend(fixtures=responses_path(), images=True, long_context=True, model="jev-1.14.0")
scored = [e for e in examples if e.split in ("validation", "test")]
offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
selection = " (selection step, not a held-out result)"
run_header(94, "Long-form visual evidence audit", backend=backend, n_examples=len(scored))
session = helpers.AuditSession(backend)
results = {}
actions, reviews = ActionLog(), ReviewQueue()

## The state

[Provenance](../../docs/glossary.md#provenance) records where evidence came from. Here that means
an equipment identifier, immutable source identifier, observation or capture date, publication
date, and image-content identity. A manifest checks registered photo identities. This establishes
internal consistency, not camera authenticity or the truth of a signed statement.

The full report is passed as source records. Python flags eligible records: the same equipment,
observation on the audit date, and publication no later than that date. A current photo must be
no more than two days old; earlier photos only explain history. The explicit snapshot rule avoids
silently treating an old maintenance entry as a current certification. Another machine's current
entry and a later-published record cannot support this asset's claim. Missing, malformed, duplicate,
stale, or unregistered photo metadata routes the entire case to review, including report-only claims.

Paths, authored labels, drawing instructions, and split names stay outside inference. References
are opaque image-content hashes. The example below shows the distant old and current statements;
the request itself retains all intervening entries.

In [ ]:
demo = next(e for e in examples if e.id == "d01")
state, questions, assets = helpers.build_request(demo.fields)
print("Claim:", state["claim"], "| asset:", state["asset_id"], "| as of:", state["as_of"])
print("Source checks:", state["photo_checks"])
for index in (2, 43):
    entry = state["history"][index]
    print(entry["source_id"], entry["observed_at"], "eligible:", entry["eligible_at_cutoff"])
    print(entry["body"])
for photo in demo.fields["photos"]:
    print(photo["role"], photo["source_id"], photo["captured_at"])
    with Image.open(photo["path"]) as opened:
        display(opened.copy())

## The questions

Each `Noul` asks whether one proposition is true, returning a probability. Visual support and
report support are separate: an exterior drawing cannot prove an internal bearing was greased.
Contradiction is separate from missing evidence: a cloth over a coupling does not show that its
guard is absent. Report support and contradiction are independent so competing current records
can both be detected. No answer chooses a source identifier, computes a date difference, or writes
an audit entry. Embedded instructions in a report are source content only.

In [ ]:
for name, question in questions.items():
    print(name + ": " + question.instructions)

## One answer up close

The demonstration's current drawing has a full coupling guard. The archived missing-guard
record describes an earlier condition. Read the typed answers before any aggregate measures.
A second unscored example has two conflicting same-day bearing records; the exterior view cannot
settle which service statement is correct.

In [ ]:
for example in [e for e in examples if e.split == "demo"]:
    result = session.decide(example)
    results[example.id] = result
    print(example.id, example.fields["claim"])
    for name, answer in result.answers.items():
        print(name)
        show_answer(answer)
    print("Composed category:", helpers.category(result.answers))

## Python's part

A [confidence gate](../../docs/glossary.md#confidence-gate) accepts a judgment only when its
certainty reaches a selected threshold. Noul has no native confidence field; we derive
`abs(2 * p - 1)` for each proposition and gate on the minimum of the four. This conservative
summary is not a calibrated probability that the entire audit is correct.

Python resolves the answers in this order: a contradiction within one evidence channel makes
the case `unknown`; otherwise a visible or report contradiction yields `contradicted`; otherwise
visible support yields `image_supported`; otherwise report support yields `report_only`; otherwise
it yields `unknown`. Visual contradiction can therefore challenge a report that claims success.
All three resolved categories pass the same gate. `unknown`, bad provenance, and low certainty
are review outcomes. The `ActionLog` records only simulated accepted entries, never maintenance
completion, permission, or equipment release. The `ReviewQueue` retains unresolved cases.

Audit entries preserve the claim, cutoff, photo identities, and eligible report identities.
These identify the sources considered; the bounded propositions do not localize a supporting span,
so the entry does not pretend to supply an exact citation.

In [ ]:
print("Fixed categories:", helpers.CATEGORIES)
for example in [e for e in examples if e.split == "demo"]:
    # Zero here illustrates the structural checks; the scored gate is selected below.
    outcome = helpers.audit(example.fields, results[example.id].answers, 0.0)
    payload = helpers.record_outcome(example, results[example.id], outcome, actions, reviews)
    print(example.id, outcome.outcome, outcome.reason)
    print(payload)
print("Simulated entries:", len(actions), "review items:", len(reviews))

## Evaluation

A [gold label](../../docs/glossary.md#gold-label) is an independently authored reference category;
readable rationales live in `labels.jsonl`. There are 12 validation cases and 12 held-out test cases,
with distinct equipment, reports, and image pixels. The two demonstrations are never scored.
These small synthetic distributions exercise the pipeline; they cannot measure Jev quality.

[Accuracy](../../docs/glossary.md#accuracy) is the fraction of categories matching the reference.
[Coverage](../../docs/glossary.md#coverage) is the fraction the policy resolves without review;
[risk](../../docs/glossary.md#risk) is the error rate among those accepted. An `unknown` reference
can match the reported category while still counting as review, not coverage.

Validation first excludes structural review cases. Among the remaining cases, the candidate gate
family is the observed minimum Noul certainties. The objective maximizes category accuracy with
at least 75% coverage of that eligible subset; ties favor the lower threshold. This constraint is
not a promise about total coverage. Freeze the chosen gate before obtaining test answers. The
report uses `evaluate_outcomes` with actual accepted flags so all review branches count.

The deterministic comparison uses the identical inputs and cutoff policy. It retrieves the top
two eligible report records by claim-word overlap (source-id tie-break), then reads pixels from
fixed apparatus regions. Orange occupancy suggests a guard; blue occupancy suggests liquid;
lexical `greased` suggests completed service. It is a real retrieval-plus-pixel baseline with
fixed rules, but it cannot reliably understand negation, occlusion, or a blue tool. It reads no
gold labels, filenames-as-labels, or synthetic model answers.

In [ ]:
validation = [e for e in examples if e.split == "validation"]
for example in validation:
    results[example.id] = session.decide(example)
threshold = helpers.choose_threshold(validation, results, labels)
val_outcomes = [helpers.audit(e.fields, results[e.id].answers, threshold) for e in validation]
val_correct = [
    o.category == labels[e.id]["category"] for e, o in zip(validation, val_outcomes, strict=True)
]
val_metrics = evaluate_outcomes([o.accepted for o in val_outcomes], val_correct)
eligible_n = sum(helpers.audit(e.fields, results[e.id].answers, 0.0).accepted for e in validation)
print(
    f"Validation N={len(validation)}, eligible N={eligible_n}; frozen gate={threshold:.4f}{selection}{check}"
)
print(
    f"Validation accepted={val_metrics.n_answered}/{val_metrics.n_total}; "
    f"coverage={val_metrics.coverage:.4f}, accepted accuracy={val_metrics.accuracy:.4f}{selection}{check}"
)

In [ ]:
test = [e for e in examples if e.split == "test"]
for example in test:
    results[example.id] = session.decide(example)
test_outcomes = [helpers.audit(e.fields, results[e.id].answers, threshold) for e in test]
gold = [labels[e.id]["category"] for e in test]
predicted = [o.category for o in test_outcomes]
accepted = [o.accepted for o in test_outcomes]
correct = [p == g for p, g in zip(predicted, gold, strict=True)]
test_metrics = evaluate_outcomes(accepted, correct)
baseline = [helpers.retrieval_pixel_baseline(e.fields) for e in test]
baseline_correct = [p == g for p, g in zip(baseline, gold, strict=True)]
baseline_metrics = evaluate_outcomes([p != "unknown" for p in baseline], baseline_correct)
for name, metric, matches in [
    ("full report + images", test_metrics, correct),
    ("retrieval + pixels", baseline_metrics, baseline_correct),
]:
    print(
        f"Test {name}, N={len(test)}: category accuracy={sum(matches) / len(matches):.4f}, "
        f"coverage={metric.coverage:.4f}, accepted accuracy={metric.accuracy:.4f}, "
        f"risk={metric.risk:.4f}, reviews={metric.n_total - metric.n_answered}{check}"
    )
for example, outcome in zip(test, test_outcomes, strict=True):
    helpers.record_outcome(example, results[example.id], outcome, actions, reviews)
print(f"Test decisions N={len(test)}{check}")
for example, expected, outcome, base in zip(test, gold, test_outcomes, baseline, strict=True):
    print(
        example.id,
        "gold=" + expected,
        "audit=" + outcome.category,
        outcome.outcome,
        "baseline=" + base,
    )

In [ ]:
matrix = confusion_matrix(gold, predicted, labels=helpers.CATEGORIES)
print(f"Test category confusion matrix, N={len(test)}; rows=gold, columns=audit{check}")
print(list(matrix.labels))
print(matrix.matrix)
plot_confusion_matrix(matrix, title=f"Test audit categories, N={len(test)}{check}")

### Inspect the accepted error

The confidently wrong held-out answer is deliberately retained. The current drawing exposes the
coupling, while the current report says the guard was fitted. A synthetic visual-support answer
misreads the image and the frozen gate accepts it. Source checks and a large context do not
ensure a correct semantic judgment. We do not retune the policy on this test failure.

In [ ]:
wrong = [e for e, o, ok in zip(test, test_outcomes, correct, strict=True) if o.accepted and not ok]
print(f"Accepted wrong test entries: {len(wrong)}, test N={len(test)}{check}")
for example in wrong:
    outcome = helpers.audit(example.fields, results[example.id].answers, threshold)
    print(
        example.id,
        example.fields["claim"],
        "accepted:",
        outcome.category,
        "reference:",
        labels[example.id]["category"],
        "certainty:",
        round(outcome.certainty, 4),
    )
    for entry in helpers.eligible_records(example.fields, helpers.load_history(example.fields)):
        print(entry["source_id"], entry["observed_at"], entry["body"])
    current = next(p for p in example.fields["photos"] if p["role"] == "current")
    with Image.open(current["path"]) as opened:
        display(opened.copy())

## What was and was not measured

The cell below prints mode and N in every run, plus returned model and capture dates when
recorded. Synthetic model identity, usage, latency, and cost remain `None`. Report word and
character counts describe local fixture text only; neither is a model token count. Replay runtime
is not inference latency. Unrecorded live exploration produces no reportable held-out claim.
A future capture should use the same `(state, questions, assets)` triples with `record`, preserve
the actual returned model/date/usage, and keep any actual measured latency in its capture record.
Missing measurements remain missing.

The result is limited to drawings with fixed viewpoints, 44-entry authored histories, a restrictive
same-day report rule, and synthetic answers. It does not measure 262K capacity, real photograph
robustness, image authenticity, real maintenance compliance, or production safety.

In [ ]:
if offline:
    print(f"Provenance: {backend.mode}. Not measured live: a pipeline check, not a Jev result.")
    print(
        f"N: {len(scored)} scored examples: validation={len(validation)}, test={len(test)}{check}"
    )
else:
    dates = ", ".join(getattr(backend, "recorded_dates", ()) or ()) or "this run"
    print(f"Provenance: {backend.mode}, returned model {backend.model}; captured {dates}")
    print(f"N: {len(scored)} scored examples: validation={len(validation)}, test={len(test)}")
    if backend.mode == "live":
        print("Unrecorded live exploration: no reportable held-out claim.")
records = [run_record(results[e.id], mode=backend.mode) for e in examples]
print("Accounting example:", records[0])
word_counts = [
    sum(len(r["body"].split()) for r in helpers.load_history(e.fields)) for e in examples
]
character_counts = [sum(len(r["body"]) for r in helpers.load_history(e.fields)) for e in examples]
print(
    "Fixture reports: N=26; body words min/max:",
    min(word_counts),
    max(word_counts),
    "; body characters min/max:",
    min(character_counts),
    max(character_counts),
)
print("Requests:", session.requests, "; images per request: 2; distinct image files: 52")
print("No measured token count, latency, cost, or 262K capacity result.")
print("Simulated entries:", len(actions), "; pending reviews:", len(reviews))

## Next steps

Use [Evidence conflict diagnosis](../47-evidence-conflict-diagnosis/) to distinguish temporal
change from a true conflict, and [Citation verification](../30-citation-verification/) to add
bounded span verification before a report presents exact supporting citations. A real study
would require licensed photographs, independently audited manifests, distinct maintenance
histories, a newly frozen validation policy, and recorded held-out inference. Compare report
lengths only after recording actual input-token usage and inference latency.